In [1]:
import os
import pandas as pd
import numpy as np

# data_monitor 폴더의 모든 .xpt 데이터 파일 임포트
folder_path_monitor = 'data_monitor'
files_monitor = [f for f in os.listdir(folder_path_monitor) if f.endswith('.xpt')]

dataframes_monitor = {}
for file in files_monitor:
    file_path = os.path.join(folder_path_monitor, file)
    # 확장자를 제외한 파일명을 키로 하여 데이터프레임 저장
    name = os.path.splitext(file)[0]
    try:
        dataframes_monitor[name] = pd.read_sas(file_path, format='xport')
        print(f"data_monitor: Successfully loaded: {file}")
    except Exception as e:
        print(f"data_monitor: Error loading {file}: {e}")

print(f"data_monitor: 총 {len(dataframes_monitor)}개의 데이터 파일을 로드했습니다.")

# data_NHANES: 클러스터링용 지정 파일/컬럼만 로드 (G/H 사이클)
NHANES_COLUMN_MAP = {
    "DEMO": ["SEQN", "RIDAGEYR"],
    "BMX": ["SEQN", "BMXBMI", "BMXWAIST"],
    "BPX": ["SEQN", "BPXSY1", "BPXDI1"],
    "GLU": ["SEQN", "LBXGLU"],
    "GHB": ["SEQN", "LBXGH"],
    "BIOPRO": ["SEQN", "LBXSCR", "LBXSATSI", "LBXSASSI", "LBXSBU", "LBXSAL"],
    "HDL": ["SEQN", "LBDHDD"],
    "TRIGLY": ["SEQN", "LBXTR"],
    "TCHOL": ["SEQN", "LBXTC"],
}

folder_path_nhanes = 'data_NHANES'
cycle_suffixes = ["G", "H"]

dataframes_nhanes = {}
for base_name, columns in NHANES_COLUMN_MAP.items():
    for suffix in cycle_suffixes:
        file = f"{base_name}_{suffix}.xpt"
        file_path = os.path.join(folder_path_nhanes, file)
        key = f"{base_name}_{suffix}"

        if not os.path.exists(file_path):
            print(f"data_NHANES: 파일 없음 - {file}")
            continue

        try:
            df = pd.read_sas(file_path, format='xport')
            df.columns = df.columns.str.upper()
            requested = [c.upper() for c in columns]
            available = [c for c in requested if c in df.columns]
            missing = sorted(set(requested) - set(available))
            if missing:
                print(f"data_NHANES: {file} 누락 컬럼 - {missing}")
            dataframes_nhanes[key] = df[available].copy()
            print(f"data_NHANES: Successfully loaded: {file} -> {available}")
        except Exception as e:
            print(f"data_NHANES: Error loading {file}: {e}")

print(f"data_NHANES: 총 {len(dataframes_nhanes)}개의 데이터 파일을 로드했습니다.")

# data_sleep 폴더의 _G 또는 _H가 포함된 .xpt 데이터 파일 임포트
sleep_data_path = 'data_sleep'
sleep_files = [f for f in os.listdir(sleep_data_path) if f.endswith('.xpt') and ('_G' in f or '_H' in f)]

sleep_dfs = {}
for file in sleep_files:
    file_path = os.path.join(sleep_data_path, file)
    # 확장자를 제외한 파일명을 키로 하여 데이터프레임 저장
    name = os.path.splitext(file)[0]
    try:
        sleep_dfs[name] = pd.read_sas(file_path, format='xport')
        print(f"Successfully loaded: {file}")
    except Exception as e:
        print(f"Error loading {file}: {e}")

# 이제 dataframes_monitor, dataframes_nhanes, sleep_dfs 딕셔너리에 모든 데이터가 저장되어 있습니다.
# 예: dataframes_monitor['PAXDAY_G'], dataframes_nhanes['DEMO_G'], sleep_dfs['SLP_G']로 접근 가능

data_monitor: Successfully loaded: PAXDAY_G.xpt
data_monitor: Successfully loaded: PAXDAY_H.xpt
data_monitor: Successfully loaded: PAXHD_G.xpt
data_monitor: Successfully loaded: PAXHD_H.xpt
data_monitor: Successfully loaded: PAXHR_G.xpt
data_monitor: Successfully loaded: PAXHR_H.xpt
data_monitor: 총 6개의 데이터 파일을 로드했습니다.
data_NHANES: Successfully loaded: DEMO_G.xpt -> ['SEQN', 'RIDAGEYR']
data_NHANES: Successfully loaded: DEMO_H.xpt -> ['SEQN', 'RIDAGEYR']
data_NHANES: Successfully loaded: BMX_G.xpt -> ['SEQN', 'BMXBMI', 'BMXWAIST']
data_NHANES: Successfully loaded: BMX_H.xpt -> ['SEQN', 'BMXBMI', 'BMXWAIST']
data_NHANES: Successfully loaded: BPX_G.xpt -> ['SEQN', 'BPXSY1', 'BPXDI1']
data_NHANES: Successfully loaded: BPX_H.xpt -> ['SEQN', 'BPXSY1', 'BPXDI1']
data_NHANES: Successfully loaded: GLU_G.xpt -> ['SEQN', 'LBXGLU']
data_NHANES: Successfully loaded: GLU_H.xpt -> ['SEQN', 'LBXGLU']
data_NHANES: Successfully loaded: GHB_G.xpt -> ['SEQN', 'LBXGH']
data_NHANES: Successfully loaded: GHB

In [2]:
# ===== Cell 1: 수면 전처리 (가속도계 전용, 설문 미사용) =====
print("\n[수면 전처리 시작 - 가속도계 전용]")

def _to_numeric(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def _circular_mean_hour(values: pd.Series):
    vals = pd.to_numeric(values, errors="coerce").dropna().to_numpy(dtype=float)
    if len(vals) == 0:
        return np.nan
    angles = vals / 24.0 * 2.0 * np.pi
    s = np.sin(angles).mean()
    c = np.cos(angles).mean()
    angle = np.arctan2(s, c)
    if angle < 0:
        angle += 2.0 * np.pi
    return float(angle / (2.0 * np.pi) * 24.0)

def _estimate_sleep_block_from_hours(hours: pd.Series):
    vals = pd.to_numeric(hours, errors="coerce").dropna().astype(int) % 24
    if vals.empty:
        return np.nan, np.nan

    mask = np.zeros(24, dtype=bool)
    mask[vals.to_numpy()] = True
    if not mask.any():
        return np.nan, np.nan

    doubled = np.concatenate([mask, mask])
    best_len = 0
    best_start = None
    run_len = 0

    for idx, is_sleep in enumerate(doubled):
        if is_sleep:
            run_len += 1
            start = idx - run_len + 1
            if start < 24 and run_len > best_len:
                best_len = run_len
                best_start = start
        else:
            run_len = 0

    if best_start is None:
        h = np.where(mask)[0]
        return float(h.min()), float((h.max() + 1) % 24)

    bedtime = float(best_start % 24)
    waketime = float((best_start + best_len) % 24)
    return bedtime, waketime


selected_keys_day = ["PAXDAY_G", "PAXDAY_H"]
selected_keys_hr = ["PAXHR_G", "PAXHR_H"]

pax_data_day = {k: v.copy() for k, v in dataframes_monitor.items() if k in selected_keys_day}
pax_data_hr = {k: v.copy() for k, v in dataframes_monitor.items() if k in selected_keys_hr}

print(f"선택된 PAX 데이터: {list(pax_data_day.keys())} + {list(pax_data_hr.keys())}")

# 1) 일 단위 필터
print("\n[일 단위 필터 적용]")
pax_day_filtered = {}
for key in selected_keys_day:
    if key not in pax_data_day:
        print(f"{key}: 없음")
        continue

    df = pax_data_day[key].copy()
    original_count = len(df)

    df = _to_numeric(df, ["SEQN", "PAXDAYWD", "PAXVMD", "PAXSWMD", "PAXQFD"])
    df = df.dropna(subset=["SEQN"])
    df["SEQN"] = df["SEQN"].astype(float)

    # 품질 필터
    mask = (df["PAXVMD"] >= 900) & (df["PAXQFD"] <= 5)
    df_f = df[mask].copy()

    # 수면분 유효 범위(너무 극단값 제거)
    if "PAXSWMD" in df_f.columns:
        df_f = df_f[df_f["PAXSWMD"].between(60, 900, inclusive="both")]

    pax_day_filtered[key] = df_f
    passed_count = len(df_f)
    ratio = (100 * passed_count / original_count) if original_count else 0
    print(f"{key}: {passed_count}/{original_count} 레코드 통과 ({ratio:.1f}%)")

# 2) 시간 단위 필터
print("\n[시간 단위 필터 적용]")
pax_hr_filtered = {}
for key in selected_keys_hr:
    if key not in pax_data_hr:
        print(f"{key}: 없음")
        continue

    df = pax_data_hr[key].copy()
    df = _to_numeric(df, ["SEQN", "PAXDAYH", "PAXDAYWH", "PAXSSNHP", "PAXVMH", "PAXSWMH", "PAXQFH"])
    df = df.dropna(subset=["SEQN", "PAXDAYH"])
    df["SEQN"] = df["SEQN"].astype(float)

    # 품질 필터(완화)
    if {"PAXVMH", "PAXQFH"}.issubset(df.columns):
        df = df[(df["PAXVMH"] >= 30) & (df["PAXQFH"] <= 10)]

    pax_hr_filtered[key] = df
    print(f"{key}: {len(df)}개 레코드 통과")

# 3) 사이클별 수면 지표 생성
sleep_cycle_frames = []
for cycle in ["G", "H"]:
    day_key = f"PAXDAY_{cycle}"
    hr_key = f"PAXHR_{cycle}"

    if day_key not in pax_day_filtered:
        print(f"\n사이클 {cycle}: PAXDAY 없음")
        continue

    day_df = pax_day_filtered[day_key].copy()
    hr_df = pax_hr_filtered.get(hr_key, pd.DataFrame()).copy()

    # ---- 일 단위: 수면시간 (SLD012/SLD013) ----
    day_df["IS_WEEKEND"] = day_df["PAXDAYWD"].isin([1, 7])  # NHANES 기준(일/토)
    day_df["SLEEP_HOURS_DAY"] = (day_df["PAXSWMD"] / 60.0).clip(3, 14)

    sld012 = (
        day_df[~day_df["IS_WEEKEND"]]
        .groupby("SEQN")["SLEEP_HOURS_DAY"]
        .mean()
        .rename("SLD012")
    )
    sld013 = (
        day_df[day_df["IS_WEEKEND"]]
        .groupby("SEQN")["SLEEP_HOURS_DAY"]
        .mean()
        .rename("SLD013")
    )
    accel_avg = (
        day_df.groupby("SEQN")["SLEEP_HOURS_DAY"]
        .mean()
        .rename("ACCELEROMETER_SLEEP_HOURS")
    )
    valid_day_count = day_df.groupby("SEQN").size().rename("VALID_DAY_COUNT")

    # ---- 시간 단위: 패턴용 취침/기상 추정 (SLQ300~330) ----
    if not hr_df.empty:
        hr_df = hr_df.sort_values(["SEQN", "PAXDAYH", "PAXSSNHP"])
        hr_df["HOUR_IDX"] = hr_df.groupby(["SEQN", "PAXDAYH"]).cumcount() % 24
        hr_df["SLEEP_ACTIVE"] = hr_df["PAXSWMH"].fillna(0) >= 30

        def _summarize_day(group: pd.DataFrame) -> pd.Series:
            bedtime, waketime = _estimate_sleep_block_from_hours(group.loc[group["SLEEP_ACTIVE"], "HOUR_IDX"])
            is_weekend = bool(group["PAXDAYWH"].iloc[0] in [1, 7])
            return pd.Series({"BEDTIME": bedtime, "WAKETIME": waketime, "IS_WEEKEND": is_weekend})

        day_time = (
            hr_df.groupby(["SEQN", "PAXDAYH"], group_keys=False)
            .apply(_summarize_day)
            .reset_index()
        )

        slq300 = (
            day_time[~day_time["IS_WEEKEND"]]
            .groupby("SEQN")["BEDTIME"]
            .apply(_circular_mean_hour)
            .rename("SLQ300")
        )
        slq310 = (
            day_time[~day_time["IS_WEEKEND"]]
            .groupby("SEQN")["WAKETIME"]
            .apply(_circular_mean_hour)
            .rename("SLQ310")
        )
        slq320 = (
            day_time[day_time["IS_WEEKEND"]]
            .groupby("SEQN")["BEDTIME"]
            .apply(_circular_mean_hour)
            .rename("SLQ320")
        )
        slq330 = (
            day_time[day_time["IS_WEEKEND"]]
            .groupby("SEQN")["WAKETIME"]
            .apply(_circular_mean_hour)
            .rename("SLQ330")
        )
        valid_night_count = day_time.groupby("SEQN").size().rename("VALID_NIGHT_COUNT")
    else:
        day_time = pd.DataFrame(columns=["SEQN", "PAXDAYH", "BEDTIME", "WAKETIME", "IS_WEEKEND"])
        slq300 = pd.Series(dtype=float, name="SLQ300")
        slq310 = pd.Series(dtype=float, name="SLQ310")
        slq320 = pd.Series(dtype=float, name="SLQ320")
        slq330 = pd.Series(dtype=float, name="SLQ330")
        valid_night_count = pd.Series(dtype=float, name="VALID_NIGHT_COUNT")

    # ---- 개인 단위 통합 ----
    seqn_union = sorted(set(day_df["SEQN"].unique()) | set(hr_df["SEQN"].unique()))
    cycle_df = pd.DataFrame({"SEQN": seqn_union})

    for s in [sld012, sld013, accel_avg, valid_day_count, slq300, slq310, slq320, slq330, valid_night_count]:
        cycle_df = cycle_df.merge(s.reset_index(), on="SEQN", how="left")

    cycle_df["CYCLE"] = cycle

    # DAILY_SLEEP: weekday/weekend 가중평균 우선, 없으면 accel 평균 사용
    both = cycle_df["SLD012"].notna() & cycle_df["SLD013"].notna()
    cycle_df["DAILY_SLEEP"] = np.where(
        both,
        (5 * cycle_df["SLD012"] + 2 * cycle_df["SLD013"]) / 7,
        cycle_df["ACCELEROMETER_SLEEP_HOURS"]
    )

    # 신뢰도(0~3): 유효 day(>=3) + 유효 night(>=3) + weekday/weekend 모두 계산 가능
    cycle_df["SLEEP_CONFIDENCE"] = (
        (cycle_df["VALID_DAY_COUNT"].fillna(0) >= 3).astype(int)
        + (cycle_df["VALID_NIGHT_COUNT"].fillna(0) >= 3).astype(int)
        + both.astype(int)
    )

    sleep_cycle_frames.append(cycle_df)

    print(f"\n사이클 {cycle}: {len(cycle_df)}명")
    print(f"  - DAILY_SLEEP 평균: {cycle_df['DAILY_SLEEP'].mean():.2f}시간")
    print(f"  - VALID_DAY_COUNT 평균: {cycle_df['VALID_DAY_COUNT'].mean():.2f}일")
    print(f"  - VALID_NIGHT_COUNT 평균: {cycle_df['VALID_NIGHT_COUNT'].mean():.2f}일")

# 4) 최종 결과
if sleep_cycle_frames:
    DAILY_SLEEP = pd.concat(sleep_cycle_frames, ignore_index=True)
    DAILY_SLEEP = DAILY_SLEEP.drop_duplicates(subset=["SEQN", "CYCLE"], keep="first")

    keep_cols = [
        "SEQN", "CYCLE",
        "SLD012", "SLD013",
        "SLQ300", "SLQ310", "SLQ320", "SLQ330",
        "ACCELEROMETER_SLEEP_HOURS",
        "DAILY_SLEEP",
        "VALID_DAY_COUNT", "VALID_NIGHT_COUNT",
        "SLEEP_CONFIDENCE",
    ]
    DAILY_SLEEP = DAILY_SLEEP[[c for c in keep_cols if c in DAILY_SLEEP.columns]]

    # downstream 호환용 이름
    sleep_final_df = DAILY_SLEEP.copy()

    print("\n[최종 데이터 통합]")
    print(f"✓ DAILY_SLEEP 생성 완료: {len(DAILY_SLEEP)}행 × {len(DAILY_SLEEP.columns)}열")
    print(f"DAILY_SLEEP 평균/표준편차: {DAILY_SLEEP['DAILY_SLEEP'].mean():.2f} / {DAILY_SLEEP['DAILY_SLEEP'].std():.2f}")
    print("사이클별 분포:")
    print(DAILY_SLEEP["CYCLE"].value_counts().sort_index())
    print("신뢰도 분포(0~3):")
    print(DAILY_SLEEP["SLEEP_CONFIDENCE"].value_counts().sort_index())
else:
    DAILY_SLEEP = pd.DataFrame()
    sleep_final_df = DAILY_SLEEP.copy()
    print("\n[최종 데이터 통합]")
    print("✗ DAILY_SLEEP 생성 실패")



[수면 전처리 시작 - 가속도계 전용]
선택된 PAX 데이터: ['PAXDAY_G', 'PAXDAY_H'] + ['PAXHR_G', 'PAXHR_H']

[일 단위 필터 적용]
PAXDAY_G: 44256/61168 레코드 통과 (72.4%)
PAXDAY_H: 48665/69018 레코드 통과 (70.5%)

[시간 단위 필터 적용]
PAXHR_G: 1305854개 레코드 통과
PAXHR_H: 1471591개 레코드 통과


C:\Users\user\AppData\Local\Temp\ipykernel_26248\2640591141.py:161: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_summarize_day)



사이클 G: 6916명
  - DAILY_SLEEP 평균: 7.53시간
  - VALID_DAY_COUNT 평균: 6.51일
  - VALID_NIGHT_COUNT 평균: 8.84일


C:\Users\user\AppData\Local\Temp\ipykernel_26248\2640591141.py:161: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_summarize_day)



사이클 H: 7771명
  - DAILY_SLEEP 평균: 7.54시간
  - VALID_DAY_COUNT 평균: 6.41일
  - VALID_NIGHT_COUNT 평균: 8.87일

[최종 데이터 통합]
✓ DAILY_SLEEP 생성 완료: 14687행 × 13열
DAILY_SLEEP 평균/표준편차: 7.54 / 1.54
사이클별 분포:
CYCLE
G    6916
H    7771
Name: count, dtype: int64
신뢰도 분포(0~3):
SLEEP_CONFIDENCE
0       76
1      806
2      655
3    13150
Name: count, dtype: int64


In [3]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

MODEL_DIR = Path("models")
FEATURES = [
    "RIDAGEYR", "BMXBMI", "BMXWAIST", "BPXSY1", "BPXDI1",
    "LBXGLU", "LBXGH", "LBXSATSI", "LBXSASSI", "LBXSCR",
    "LBXSBU", "LBXSAL", "LBDHDD", "LBXTR", "LBXTC",
]
AGE_MIN = 20
AGE_MAX = 70

with open(MODEL_DIR / "imputer.pkl", "rb") as f:
    imputer = pickle.load(f)
with open(MODEL_DIR / "cluster_scaler.pkl", "rb") as f:
    cluster_scaler = pickle.load(f)
with open(MODEL_DIR / "kmeans.pkl", "rb") as f:
    kmeans = pickle.load(f)
with open(MODEL_DIR / "label_map.pkl", "rb") as f:
    label_map = pickle.load(f)

feature_tables = {"G": [], "H": []}
for name, df in dataframes_nhanes.items():
    suffix = name.rsplit("_", 1)[-1] if "_" in name else ""
    if suffix not in feature_tables:
        continue
    if "SEQN" not in df.columns:
        continue

    available = [c for c in FEATURES if c in df.columns]
    if not available:
        continue

    slim = df[["SEQN", *available]].copy()
    slim["SEQN"] = pd.to_numeric(slim["SEQN"], errors="coerce")
    slim = slim.dropna(subset=["SEQN"])
    for col in available:
        slim[col] = pd.to_numeric(slim[col], errors="coerce")
    slim = slim.groupby("SEQN", as_index=False).first()
    feature_tables[suffix].append(slim)

cluster_lookup_frames = []
for cycle, tables in feature_tables.items():
    if not tables:
        continue

    merged = tables[0]
    for t in tables[1:]:
        merged = merged.merge(t, on="SEQN", how="outer", suffixes=("", "_dup"))
        dup_cols = [c for c in merged.columns if c.endswith("_dup")]
        for dup_col in dup_cols:
            base_col = dup_col[:-4]
            merged[base_col] = merged[base_col].combine_first(merged[dup_col])
        if dup_cols:
            merged = merged.drop(columns=dup_cols)

    if "RIDAGEYR" in merged.columns:
        merged["RIDAGEYR"] = pd.to_numeric(merged["RIDAGEYR"], errors="coerce")
        merged = merged[(merged["RIDAGEYR"] >= AGE_MIN) & (merged["RIDAGEYR"] < AGE_MAX)].copy()

    if merged.empty:
        continue

    X_raw = merged.reindex(columns=FEATURES)
    X_imputed = pd.DataFrame(
        imputer.transform(X_raw),
        columns=FEATURES,
        index=merged.index,
    )
    X_scaled = cluster_scaler.transform(X_imputed)
    X_scaled_df = pd.DataFrame(X_scaled, columns=FEATURES, index=merged.index)
    pred_raw = kmeans.predict(X_scaled_df)
    pred_cluster = pd.Series(pred_raw, index=merged.index).map(label_map).astype("Int64")

    lookup = pd.DataFrame({
        "SEQN": merged["SEQN"].astype(float),
        "CYCLE": cycle,
        "Cluster": pred_cluster,
    })
    cluster_lookup_frames.append(lookup)

if cluster_lookup_frames:
    cluster_lookup = pd.concat(cluster_lookup_frames, ignore_index=True)
else:
    cluster_lookup = pd.DataFrame(columns=["SEQN", "CYCLE", "Cluster"])

for name, df in dataframes_nhanes.items():
    suffix = name.rsplit("_", 1)[-1] if "_" in name else ""
    result = df.copy()
    cluster_like_cols = [c for c in result.columns if c.startswith("Cluster")]
    if cluster_like_cols:
        result = result.drop(columns=cluster_like_cols)

    if suffix in {"G", "H"} and "SEQN" in result.columns and not cluster_lookup.empty:
        result["SEQN"] = pd.to_numeric(result["SEQN"], errors="coerce")
        result = result.merge(
            cluster_lookup[cluster_lookup["CYCLE"] == suffix][["SEQN", "Cluster"]],
            on="SEQN",
            how="left",
        )
    else:
        result["Cluster"] = pd.NA
    dataframes_nhanes[name] = result

print("Cluster 컬럼 추가 완료")
print({k: int(v["Cluster"].notna().sum()) for k, v in dataframes_nhanes.items() if "Cluster" in v.columns})

Cluster 컬럼 추가 완료
{'DEMO_G': 4677, 'DEMO_H': 4868, 'BMX_G': 4500, 'BMX_H': 4721, 'BPX_G': 4500, 'BPX_H': 4721, 'GLU_G': 2205, 'GLU_H': 2239, 'GHB_G': 4500, 'GHB_H': 4721, 'BIOPRO_G': 4500, 'BIOPRO_H': 4721, 'HDL_G': 4500, 'HDL_H': 4721, 'TRIGLY_G': 2205, 'TRIGLY_H': 2239, 'TCHOL_G': 4500, 'TCHOL_H': 4721}


In [4]:
# dataframes_nhanes(dict) -> 하나의 통합 데이터프레임 (리스크 노트북 전처리 동일 적용)
cycle_merged_frames = []
AGE_MIN = 20
AGE_MAX = 70

for cycle in ["G", "H"]:
    cycle_tables = []
    for name, df in dataframes_nhanes.items():
        suffix = name.rsplit("_", 1)[-1] if "_" in name else ""
        if suffix != cycle:
            continue

        tmp = df.copy()
        if "SEQN" not in tmp.columns:
            continue
        tmp["SEQN"] = pd.to_numeric(tmp["SEQN"], errors="coerce")
        tmp = tmp.dropna(subset=["SEQN"])
        cycle_tables.append(tmp)

    if not cycle_tables:
        continue

    merged_cycle = cycle_tables[0]
    for table_df in cycle_tables[1:]:
        merged_cycle = merged_cycle.merge(table_df, on="SEQN", how="outer", suffixes=("", "_dup"))

        dup_cols = [c for c in merged_cycle.columns if c.endswith("_dup")]
        for dup_col in dup_cols:
            base_col = dup_col[:-4]
            if base_col in merged_cycle.columns:
                merged_cycle[base_col] = merged_cycle[base_col].combine_first(merged_cycle[dup_col])
            else:
                merged_cycle[base_col] = merged_cycle[dup_col]
        if dup_cols:
            merged_cycle = merged_cycle.drop(columns=dup_cols)

    merged_cycle["CYCLE"] = cycle
    cycle_merged_frames.append(merged_cycle)

if cycle_merged_frames:
    nhanes_result_df = pd.concat(cycle_merged_frames, ignore_index=True, sort=False)

    if "RIDAGEYR" in nhanes_result_df.columns:
        nhanes_result_df["RIDAGEYR"] = pd.to_numeric(nhanes_result_df["RIDAGEYR"], errors="coerce")
        nhanes_result_df = nhanes_result_df[
            (nhanes_result_df["RIDAGEYR"] >= AGE_MIN) & (nhanes_result_df["RIDAGEYR"] < AGE_MAX)
        ].copy()

    # models/imputer.pkl 기반 결측치 처리
    try:
        import pickle
        from pathlib import Path

        model_imputer = imputer if "imputer" in globals() else None
        if model_imputer is None:
            with open(Path("models") / "imputer.pkl", "rb") as f:
                model_imputer = pickle.load(f)

        if "FEATURES" in globals() and isinstance(FEATURES, list):
            impute_columns = [c for c in FEATURES if c in nhanes_result_df.columns]
        else:
            feature_names = getattr(model_imputer, "feature_names_in_", [])
            impute_columns = [c for c in feature_names if c in nhanes_result_df.columns]

        if impute_columns:
            impute_input = nhanes_result_df.reindex(columns=impute_columns).apply(pd.to_numeric, errors="coerce")
            nhanes_result_df.loc[:, impute_columns] = model_imputer.transform(impute_input)
            print(f"imputer 적용 완료: {len(impute_columns)}개 컬럼 결측치 처리")
        else:
            print("imputer 적용 스킵: 처리 대상 피처 컬럼이 없습니다.")
    except Exception as e:
        print(f"imputer 적용 실패: {e}")

    if {"CYCLE", "SEQN"}.issubset(nhanes_result_df.columns):
        nhanes_result_df = nhanes_result_df.sort_values(["CYCLE", "SEQN"]).reset_index(drop=True)
else:
    nhanes_result_df = pd.DataFrame()

print(f"nhanes_result_df 생성 완료: {nhanes_result_df.shape}")
nhanes_result_df.head()

imputer 적용 완료: 15개 컬럼 결측치 처리
nhanes_result_df 생성 완료: (9545, 18)


,SEQN,RIDAGEYR,Cluster,BMXBMI,BMXWAIST,BPXSY1,BPXDI1,LBXGLU,LBXGH,LBXSCR,LBXSATSI,LBXSASSI,LBXSBU,LBXSAL,LBDHDD,LBXTR,LBXTC,CYCLE
0,62161.0,22.0,1,23.3,81.0,110.000000,82.000000,92.000000,5.1,0.91,19.0,25.0,14.0,4.8,41.0,84.000000,168.0,G
1,62164.0,44.0,1,23.2,80.1,116.000000,56.000000,82.000000,4.9,0.89,29.0,37.0,5.0,3.7,28.0,56.000000,190.0,G
2,62169.0,21.0,1,20.1,69.6,124.000000,80.000000,107.000000,5.4,0.87,19.0,17.0,16.0,4.4,43.0,78.000000,132.0,G
3,62172.0,43.0,1,33.3,120.4,100.000000,70.000000,104.000000,5.6,0.68,17.0,15.0,8.0,4.4,73.0,141.000000,169.0,G
4,62176.0,34.0,1,23.3,89.4,115.068835,68.886337,101.110054,5.4,0.65,17.0,22.0,11.0,4.5,55.0,94.762841,171.0,G


In [5]:
# nhanes_result_df + sleep_final_df 통합 후 final_df 저장
if "nhanes_result_df" not in globals() or nhanes_result_df.empty:
    raise ValueError("nhanes_result_df가 없습니다. 통합 데이터프레임 생성 셀을 먼저 실행하세요.")
if "sleep_final_df" not in globals() or sleep_final_df.empty:
    raise ValueError("sleep_final_df가 없습니다. 수면 전처리 셀을 먼저 실행하세요.")

nhanes_base = nhanes_result_df.copy()
sleep_base = sleep_final_df.copy()

if not {"SEQN", "CYCLE", "Cluster"}.issubset(nhanes_base.columns):
    raise ValueError("nhanes_result_df에 SEQN/CYCLE/Cluster 컬럼이 필요합니다.")
if not {"SEQN", "CYCLE"}.issubset(sleep_base.columns):
    raise ValueError("sleep_final_df에 SEQN/CYCLE 컬럼이 필요합니다.")

nhanes_base["SEQN"] = pd.to_numeric(nhanes_base["SEQN"], errors="coerce")
nhanes_base["Cluster"] = pd.to_numeric(nhanes_base["Cluster"], errors="coerce").astype("Int64")
nhanes_base["CYCLE"] = nhanes_base["CYCLE"].astype(str)
nhanes_base = nhanes_base.dropna(subset=["SEQN", "Cluster", "CYCLE"]).copy()

sleep_base["SEQN"] = pd.to_numeric(sleep_base["SEQN"], errors="coerce")
sleep_base["CYCLE"] = sleep_base["CYCLE"].astype(str)
sleep_base = sleep_base.dropna(subset=["SEQN", "CYCLE"]).copy()

final_df = nhanes_base.merge(
    sleep_base,
    on=["SEQN", "CYCLE"],
    how="left",
    suffixes=("", "_sleep"),
)
final_df = final_df.dropna().copy()
final_df = final_df.sort_values(["Cluster", "CYCLE", "SEQN"]).reset_index(drop=True)

print("통합 완료")
print(f"final_df 크기: {final_df.shape}")
print("Cluster 분포:", final_df["Cluster"].value_counts(dropna=False).sort_index().to_dict())

final_df.head()

통합 완료
final_df 크기: (7382, 29)
Cluster 분포: {np.int64(1): 3568, np.int64(2): 3480, np.int64(3): 334}


,SEQN,RIDAGEYR,Cluster,BMXBMI,BMXWAIST,BPXSY1,BPXDI1,LBXGLU,LBXGH,LBXSCR,...,SLD013,SLQ300,SLQ310,SLQ320,SLQ330,ACCELEROMETER_SLEEP_HOURS,DAILY_SLEEP,VALID_DAY_COUNT,VALID_NIGHT_COUNT,SLEEP_CONFIDENCE
0,62161.0,22.0,1,23.3,81.0,110.0,82.0,92.00000,5.1,0.91,...,7.058333,2.258009e+01,6.391116,24.000000,7.922216,8.002381,8.002381,7.0,9.0,3.0
1,62164.0,44.0,1,23.2,80.1,116.0,56.0,82.00000,4.9,0.89,...,9.050000,2.360337e+01,5.427889,2.365311,7.824520,7.095238,7.095238,7.0,9.0,3.0
2,62169.0,21.0,1,20.1,69.6,124.0,80.0,107.00000,5.4,0.87,...,7.116667,2.284710e+00,9.447378,4.000000,11.000000,7.033333,7.033333,7.0,9.0,3.0
3,62172.0,43.0,1,33.3,120.4,100.0,70.0,104.00000,5.6,0.68,...,6.633333,1.097584e-16,3.000000,0.000000,6.000000,6.516667,6.466667,2.0,9.0,2.0
4,62180.0,35.0,1,27.9,103.1,108.0,62.0,99.76677,5.3,0.82,...,7.116667,2.333248e+01,7.000000,1.377142,7.000000,7.485417,7.467857,8.0,9.0,3.0


In [6]:
# 클러스터별 샘플링 개수(원하는 숫자로 수정)
cluster_1 = 65
cluster_2 = 26
cluster_3 = 9
seed = 42

sample_size_by_cluster = {
    1: int(cluster_1),
    2: int(cluster_2),
    3: int(cluster_3),
}

if "final_df" not in globals() or final_df.empty:
    raise ValueError("final_df가 없습니다. 통합 셀을 먼저 실행하세요.")

if not {"SEQN", "CYCLE", "Cluster"}.issubset(final_df.columns):
    raise ValueError("final_df에 SEQN/CYCLE/Cluster 컬럼이 필요합니다.")

sampling_base = final_df.copy()
sampling_base["SEQN"] = pd.to_numeric(sampling_base["SEQN"], errors="coerce")
sampling_base["Cluster"] = pd.to_numeric(sampling_base["Cluster"], errors="coerce").astype("Int64")
sampling_base["CYCLE"] = sampling_base["CYCLE"].astype(str)
sampling_base = sampling_base.dropna(subset=["SEQN", "Cluster", "CYCLE"]).copy()

sampled_parts = []
for cluster_label, requested_size in sample_size_by_cluster.items():
    cluster_df = sampling_base[sampling_base["Cluster"] == cluster_label].copy()
    if cluster_df.empty or requested_size <= 0:
        continue

    actual_size = min(requested_size, len(cluster_df))
    sampled_cluster = cluster_df.sample(n=actual_size, random_state=seed, replace=False)
    sampled_parts.append(sampled_cluster)

if sampled_parts:
    sampled_sleep_merged_df = pd.concat(sampled_parts, ignore_index=True)
    sampled_sleep_merged_df = sampled_sleep_merged_df.sort_values(["Cluster", "CYCLE", "SEQN"]).reset_index(drop=True)
else:
    sampled_sleep_merged_df = pd.DataFrame(columns=sampling_base.columns)

print("샘플링 완료")
print("요청 수:", sample_size_by_cluster)
print("seed:", seed)
print("실제 추출 수:", sampled_sleep_merged_df["Cluster"].value_counts().sort_index().to_dict())
print(f"샘플링 데이터 크기: {sampled_sleep_merged_df.shape}")

sampled_sleep_merged_df.head()

샘플링 완료
요청 수: {1: 65, 2: 26, 3: 9}
seed: 42
실제 추출 수: {np.int64(1): 65, np.int64(2): 26, np.int64(3): 9}
샘플링 데이터 크기: (100, 29)


,SEQN,RIDAGEYR,Cluster,BMXBMI,BMXWAIST,BPXSY1,BPXDI1,LBXGLU,LBXGH,LBXSCR,...,SLD013,SLQ300,SLQ310,SLQ320,SLQ330,ACCELEROMETER_SLEEP_HOURS,DAILY_SLEEP,VALID_DAY_COUNT,VALID_NIGHT_COUNT,SLEEP_CONFIDENCE
0,62531.0,52.0,1,33.3,101.7,92.0,68.0,104.594037,5.726984,0.766529,...,9.091667,0.644191,5.344431,0.725442,4.340261,7.579167,7.651190,8.0,9.0,3.0
1,62674.0,33.0,1,21.5,81.7,106.0,70.0,98.000000,5.300000,0.810000,...,8.641667,23.662465,6.700349,1.365311,5.667518,7.911905,7.911905,7.0,9.0,3.0
2,63133.0,24.0,1,29.8,98.4,102.0,58.0,100.000000,5.600000,0.950000,...,6.208333,1.000000,6.708092,3.077784,6.975920,6.273810,6.273810,7.0,9.0,3.0
3,63429.0,21.0,1,22.6,73.7,116.0,64.0,101.089691,5.500000,0.780000,...,9.258333,3.304858,9.659739,2.500000,12.000000,7.476190,7.476190,7.0,9.0,3.0
4,63551.0,54.0,1,28.6,94.3,126.0,76.0,94.000000,5.100000,0.800000,...,7.766667,0.988242,6.577784,1.000000,9.000000,6.933333,6.933333,7.0,9.0,3.0


In [7]:
from pathlib import Path

import pandas as pd

# -----------------------------
# 남길 컬럼 정의
# -----------------------------
sleep_cols = ["SLD012", "SLD013", "DAILY_SLEEP", "SLQ300", "SLQ310", "SLQ320", "SLQ330"]
step_cols_keep = ["avg_daily_steps"]
extra_cols = ["RIDAGEYR"]

# -----------------------------
# 입력 데이터 확인
# -----------------------------
if "sampled_sleep_merged_df" not in globals() or sampled_sleep_merged_df.empty:
    raise ValueError("sampled_sleep_merged_df가 없습니다. 이전 샘플링 셀을 먼저 실행하세요.")

base_df = sampled_sleep_merged_df.copy()
required_id_cols = {"SEQN", "CYCLE", "Cluster"}
if not required_id_cols.issubset(base_df.columns):
    raise ValueError("sampled_sleep_merged_df에 SEQN/CYCLE/Cluster 컬럼이 필요합니다.")

base_df["SEQN"] = pd.to_numeric(base_df["SEQN"], errors="coerce")
base_df["CYCLE"] = base_df["CYCLE"].astype(str)
base_df["Cluster"] = pd.to_numeric(base_df["Cluster"], errors="coerce").astype("Int64")
base_df = base_df.dropna(subset=["SEQN", "CYCLE", "Cluster"]).copy()

# -----------------------------
# 걸음수 컬럼 확보(avg_daily_steps)
# -----------------------------
if "avg_daily_steps" not in base_df.columns:
    step_csv_path = Path("data_steps") / "csv" / "nhanes_1440_scsslsteps.csv"
    if not step_csv_path.exists():
        raise FileNotFoundError(f"걸음수 파일을 찾을 수 없습니다: {step_csv_path}")

    step_raw = pd.read_csv(step_csv_path)
    if "SEQN" not in step_raw.columns:
        raise ValueError("걸음수 파일에 SEQN 컬럼이 없습니다.")

    step_cols = [c for c in step_raw.columns if c.startswith("step_min_")]
    if not step_cols:
        step_cols = [c for c in step_raw.columns if c.startswith("min_")]
    if not step_cols:
        step_cols = [c for c in step_raw.columns if c.startswith("step_")]
    if not step_cols:
        candidate_cols = [c for c in step_raw.columns if c not in {"SEQN", "PAXDAYM", "PAXDAYWM"}]
        numeric_candidates = []
        for c in candidate_cols:
            converted = pd.to_numeric(step_raw[c], errors="coerce")
            if converted.notna().mean() >= 0.7:
                numeric_candidates.append(c)
        step_cols = numeric_candidates

    if not step_cols:
        raise ValueError("걸음수 파일에서 avg_daily_steps 계산용 수치형 컬럼을 찾지 못했습니다.")

    for c in step_cols:
        step_raw[c] = pd.to_numeric(step_raw[c], errors="coerce")

    step_raw["daily_total_steps"] = step_raw[step_cols].sum(axis=1, skipna=True)
    step_score_df = (
        step_raw.groupby("SEQN", as_index=False)["daily_total_steps"]
        .mean()
        .rename(columns={"daily_total_steps": "avg_daily_steps"})
    )
    step_score_df["SEQN"] = pd.to_numeric(step_score_df["SEQN"], errors="coerce")

    base_df = base_df.merge(step_score_df, on="SEQN", how="left")

# -----------------------------
# 최종: 지정 컬럼만 유지
# -----------------------------
missing_extra_cols = [c for c in extra_cols if c not in base_df.columns]
if missing_extra_cols:
    raise ValueError(f"필수 추가 컬럼이 없습니다: {missing_extra_cols}")

existing_sleep_cols = [c for c in sleep_cols if c in base_df.columns]
existing_step_cols = [c for c in step_cols_keep if c in base_df.columns]
selected_cols = ["SEQN", "CYCLE", "Cluster"] + extra_cols + existing_sleep_cols + existing_step_cols

if len(selected_cols) <= 4:
    raise ValueError("수면/걸음수 컬럼이 없어 최종 컬럼 구성이 불가능합니다.")

# RIDAGEYR(나이) 및 원 컬럼 값 고정: SEQN+CYCLE 기준 첫 원값 유지
seqn_score_samples_df = (
    base_df[selected_cols]
    .sort_values(["SEQN", "CYCLE"])
    .groupby(["SEQN", "CYCLE"], as_index=False)
    .first()
)

# 하위 셀 호환용 변수명 유지
seqn_score_param_df = seqn_score_samples_df.copy()

print("요청 컬럼만 유지 완료")
print("RIDAGEYR 포함 원 컬럼 값 고정 완료(SEQN+CYCLE 기준)")
print(f"최종 컬럼: {selected_cols}")
print(f"결과 크기: {seqn_score_samples_df.shape}")

seqn_score_samples_df.head()

요청 컬럼만 유지 완료
RIDAGEYR 포함 원 컬럼 값 고정 완료(SEQN+CYCLE 기준)
최종 컬럼: ['SEQN', 'CYCLE', 'Cluster', 'RIDAGEYR', 'SLD012', 'SLD013', 'DAILY_SLEEP', 'SLQ300', 'SLQ310', 'SLQ320', 'SLQ330', 'avg_daily_steps']
결과 크기: (100, 12)


C:\Users\user\AppData\Local\Temp\ipykernel_26248\1896550813.py:60: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  step_raw["daily_total_steps"] = step_raw[step_cols].sum(axis=1, skipna=True)


,SEQN,CYCLE,Cluster,RIDAGEYR,SLD012,SLD013,DAILY_SLEEP,SLQ300,SLQ310,SLQ320,SLQ330,avg_daily_steps
0,62467.0,G,2,65.0,9.396667,9.066667,9.302381,23.399448,7.806781,7.232186e-17,7.340261,13760.555556
1,62531.0,G,1,52.0,7.075000,9.091667,7.651190,0.644191,5.344431,7.254421e-01,4.340261,5886.666667
2,62674.0,G,1,33.0,7.620000,8.641667,7.911905,23.662465,6.700349,1.365311e+00,5.667518,6610.666667
3,62935.0,G,2,51.0,6.411111,7.683333,6.774603,22.450038,5.157875,1.675480e+00,7.306781,9581.555556
4,63133.0,G,1,24.0,6.300000,6.208333,6.273810,1.000000,6.708092,3.077784e+00,6.975920,14744.222222


In [ ]:
# seqn_score_samples_df 기반 샘플링 (seed=42, 30회)
# 기존 로직(전역 평균/표준편차 기반 normal/lognormal 샘플링)은 아래에 주석으로 보존합니다.
# ------------------------------------------------------------
# [기존 로직 요약]
# 1) 각 컬럼의 전체 평균/표준편차로 분포 파라미터 계산
# 2) avg_daily_steps는 lognormal, 나머지는 normal로 고정
# 3) 개인별 차이를 분포 평균에 반영하지 않고 동일 분포에서 30회 샘플링
# ------------------------------------------------------------

import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.special import gamma
from scipy.stats import nbinom, skewnorm, weibull_min

if "seqn_score_samples_df" not in globals() or seqn_score_samples_df.empty:
    raise ValueError("seqn_score_samples_df가 없습니다. 이전 셀을 먼저 실행하세요.")

base_df = seqn_score_samples_df.copy()
id_cols = ["SEQN", "CYCLE", "Cluster", "RIDAGEYR"]
existing_id_cols = [c for c in id_cols if c in base_df.columns]
if "RIDAGEYR" not in base_df.columns:
    raise ValueError("RIDAGEYR 컬럼이 필요합니다. 나이대별 분포 파라미터 매핑에 사용됩니다.")

# 샘플링 대상 컬럼
candidate_cols = [
    "SLD012", "SLD013", "DAILY_SLEEP",  # sleep_time
    "SLQ300", "SLQ310", "SLQ320", "SLQ330",  # sleep_pattern
    "avg_daily_steps",  # step
]
target_cols = [c for c in candidate_cols if c in base_df.columns]
if not target_cols:
    raise ValueError("샘플링할 수치형 컬럼이 없습니다.")

for col in target_cols:
    base_df[col] = pd.to_numeric(base_df[col], errors="coerce")
base_df["RIDAGEYR"] = pd.to_numeric(base_df["RIDAGEYR"], errors="coerce")

# percentile 파라미터 파일 로드 (우선순위: percentile.csv -> percentile_1.csv)
param_path = Path("percentile.csv")
if not param_path.exists():
    fallback = Path("percentile_1.csv")
    if fallback.exists():
        param_path = fallback
    else:
        raise FileNotFoundError("percentile.csv 또는 percentile_1.csv를 찾을 수 없습니다.")

param_df = pd.read_csv(param_path)
required_cols = {"domain", "distribution", "age_group"}
if not required_cols.issubset(param_df.columns):
    raise ValueError("percentile 파일에 domain/distribution/age_group 컬럼이 필요합니다.")

# 파라미터 컬럼 보정(없으면 생성)
for col in ["param_s", "param_n", "param_p", "param_mu", "param_shape", "param_loc", "param_scale", "fit_params"]:
    if col not in param_df.columns:
        param_df[col] = np.nan

# fit_params(JSON)에서 파라미터 복원

def _json_get(params_text, keys):
    if pd.isna(params_text):
        return np.nan
    try:
        obj = json.loads(params_text)
    except Exception:
        return np.nan
    for k in keys:
        if k in obj and obj[k] is not None:
            try:
                return float(obj[k])
            except Exception:
                continue
    return np.nan

for idx, row in param_df.iterrows():
    if pd.isna(row.get("param_shape", np.nan)):
        param_df.at[idx, "param_shape"] = _json_get(row.get("fit_params", np.nan), ["shape", "s"])
    if pd.isna(row.get("param_loc", np.nan)):
        param_df.at[idx, "param_loc"] = _json_get(row.get("fit_params", np.nan), ["loc"])
    if pd.isna(row.get("param_scale", np.nan)):
        param_df.at[idx, "param_scale"] = _json_get(row.get("fit_params", np.nan), ["scale"])
    if pd.isna(row.get("param_n", np.nan)):
        param_df.at[idx, "param_n"] = _json_get(row.get("fit_params", np.nan), ["n"])
    if pd.isna(row.get("param_p", np.nan)):
        param_df.at[idx, "param_p"] = _json_get(row.get("fit_params", np.nan), ["p"])
    if pd.isna(row.get("param_mu", np.nan)):
        param_df.at[idx, "param_mu"] = _json_get(row.get("fit_params", np.nan), ["mu"])

# age_group별 1개 파라미터 행으로 축약
param_df = param_df.sort_values(["domain", "distribution", "age_group"]).copy()
param_df = param_df.drop_duplicates(subset=["domain", "distribution", "age_group"], keep="first").reset_index(drop=True)


def _age_to_group_label(age):
    if pd.isna(age):
        return np.nan
    decade = int(float(age) // 10 * 10)
    return f"{decade}대"


def _extract_decade(label):
    if pd.isna(label):
        return None
    m = re.search(r"(\d+)", str(label))
    return int(m.group(1)) if m else None


base_df["AGE_GROUP_LABEL"] = base_df["RIDAGEYR"].apply(_age_to_group_label)

# 컬럼 -> 도메인/분포 매핑
col_dist_map = {
    "SLD012": ("sleep_time", "skewnorm"),
    "SLD013": ("sleep_time", "skewnorm"),
    "DAILY_SLEEP": ("sleep_time", "skewnorm"),
    "SLQ300": ("sleep_pattern", "weibull_min"),
    "SLQ310": ("sleep_pattern", "weibull_min"),
    "SLQ320": ("sleep_pattern", "weibull_min"),
    "SLQ330": ("sleep_pattern", "weibull_min"),
    "avg_daily_steps": ("step", "nbinom"),
}

# domain/distribution/age_group 키별 파라미터 딕셔너리 생성
param_lookup = {}
for _, r in param_df.iterrows():
    key = (str(r["domain"]), str(r["distribution"]), str(r["age_group"]))
    param_lookup[key] = {
        "param_s": pd.to_numeric(pd.Series([r.get("param_s", np.nan)]), errors="coerce").iloc[0],
        "param_shape": pd.to_numeric(pd.Series([r.get("param_shape", np.nan)]), errors="coerce").iloc[0],
        "param_loc": pd.to_numeric(pd.Series([r.get("param_loc", np.nan)]), errors="coerce").iloc[0],
        "param_scale": pd.to_numeric(pd.Series([r.get("param_scale", np.nan)]), errors="coerce").iloc[0],
        "param_n": pd.to_numeric(pd.Series([r.get("param_n", np.nan)]), errors="coerce").iloc[0],
        "param_p": pd.to_numeric(pd.Series([r.get("param_p", np.nan)]), errors="coerce").iloc[0],
        "param_mu": pd.to_numeric(pd.Series([r.get("param_mu", np.nan)]), errors="coerce").iloc[0],
    }

# age_group fallback 매핑(예: 20대 매칭 실패 시 숫자 decade 기반 근사)
available_keys = list(param_lookup.keys())


def _find_params(domain, dist, age_label):
    exact = param_lookup.get((domain, dist, age_label))
    if exact is not None:
        return exact

    target_decade = _extract_decade(age_label)
    candidates = [
        (k, v) for k, v in param_lookup.items()
        if k[0] == domain and k[1] == dist
    ]
    if not candidates:
        return None

    if target_decade is None:
        return candidates[0][1]

    best = min(
        candidates,
        key=lambda kv: abs((_extract_decade(kv[0][2]) if _extract_decade(kv[0][2]) is not None else 9999) - target_decade),
    )
    return best[1]


# 개인별 평균으로 shift한 분포에서 30개 샘플링
N_SAMPLES = 30
SEED = 42
rng = np.random.default_rng(SEED)

expanded_id = base_df[existing_id_cols].copy().reset_index(drop=True)
expanded_id = expanded_id.loc[expanded_id.index.repeat(N_SAMPLES)].reset_index(drop=True)
expanded_id.insert(0, "sample_idx", np.tile(np.arange(1, N_SAMPLES + 1), len(base_df)))

sampled_values = {}
failed_cols = []

for col in target_cols:
    domain, dist = col_dist_map[col]
    samples = np.full(len(expanded_id), np.nan, dtype=float)

    for row_idx, row in base_df.reset_index(drop=True).iterrows():
        mean_target = row[col]
        age_label = row["AGE_GROUP_LABEL"]

        if pd.isna(mean_target):
            continue

        params = _find_params(domain, dist, age_label)
        if params is None:
            continue

        start = row_idx * N_SAMPLES
        end = start + N_SAMPLES

        try:
            if dist == "skewnorm":
                shape = params["param_shape"] if pd.notna(params["param_shape"]) else params["param_s"]
                scale = params["param_scale"]
                if pd.isna(shape) or pd.isna(scale) or scale <= 0:
                    continue
                delta = shape / np.sqrt(1.0 + shape ** 2)
                mean_offset = scale * delta * np.sqrt(2.0 / np.pi)
                loc_shift = float(mean_target - mean_offset)
                draw = skewnorm.rvs(shape, loc=loc_shift, scale=scale, size=N_SAMPLES, random_state=rng)
                samples[start:end] = draw

            elif dist == "weibull_min":
                shape = params["param_shape"] if pd.notna(params["param_shape"]) else params["param_s"]
                scale = params["param_scale"]
                base_loc = params["param_loc"] if pd.notna(params["param_loc"]) else 0.0
                if pd.isna(shape) or pd.isna(scale) or shape <= 0 or scale <= 0:
                    continue
                base_mean_component = scale * gamma(1.0 + 1.0 / shape)
                loc_shift = float(mean_target - base_mean_component)
                draw = weibull_min.rvs(shape, loc=loc_shift, scale=scale, size=N_SAMPLES, random_state=rng)
                samples[start:end] = draw

            elif dist == "nbinom":
                n = params["param_n"]
                p = params["param_p"]
                if pd.isna(n) or pd.isna(p) or n <= 0 or p <= 0 or p >= 1:
                    continue
                base_mean_component = n * (1.0 - p) / p
                loc_shift = float(mean_target - base_mean_component)
                draw = nbinom.rvs(n, p, loc=loc_shift, size=N_SAMPLES, random_state=rng)
                draw = np.clip(draw, 0, None)
                samples[start:end] = draw
        except Exception:
            continue

    sampled_values[col] = samples
    if np.isnan(samples).all():
        failed_cols.append(col)

sampled_df = pd.DataFrame(sampled_values)
normal_sampled_df = pd.concat([expanded_id, sampled_df], axis=1)

print(f"식별 컬럼 포함: {existing_id_cols}")
print(f"샘플링 대상 컬럼 수: {len(target_cols)}")
print(f"랜덤 시드: {SEED}, 원본 행 수: {len(base_df)}, 행당 샘플 수: {N_SAMPLES}")
print(f"결과 데이터프레임 크기: {normal_sampled_df.shape}")
print(f"파라미터 파일: {param_path}")
if failed_cols:
    print("주의: 파라미터 매핑/샘플링 실패 컬럼 ->", failed_cols)
print(normal_sampled_df.head())

식별 컬럼 포함: ['SEQN', 'CYCLE', 'Cluster', 'RIDAGEYR']
샘플링 대상 컬럼 수: 8
랜덤 시드: 42, 원본 행 수: 100, 행당 샘플 수: 30
결과 데이터프레임 크기: (3000, 13)
avg_daily_steps 분포: lognormal
   sample_idx     SEQN CYCLE  Cluster  RIDAGEYR    SLD012     SLD013  \
0           1  62467.0     G        2      65.0  7.823845  10.345670   
1           2  62467.0     G        2      65.0  5.537973   9.283175   
2           3  62467.0     G        2      65.0  8.581553  11.703017   
3           4  62467.0     G        2      65.0  8.904729   5.051738   
4           5  62467.0     G        2      65.0  3.989267   7.479222   

   DAILY_SLEEP     SLQ300    SLQ310     SLQ320     SLQ330  avg_daily_steps  
0     7.853568  11.693496  6.278536  17.181995  10.250052      6046.687788  
1     7.414852  -1.075316  7.435429  17.494311  13.346798      4633.644562  
2     8.860204  11.667438  3.367829  15.483315   5.965836      8359.831410  
3     8.454809  -1.365778  2.908235  12.536874   6.370728      7271.501469  
4     2.794703  23.021420

In [9]:
if "final_df" not in globals() or final_df.empty:
    raise ValueError("final_df가 없습니다. final_df 생성 셀을 먼저 실행하세요.")

if "Cluster" not in final_df.columns:
    raise ValueError("final_df에 Cluster 컬럼이 없습니다.")

cluster_counts = (
    final_df["Cluster"]
    .value_counts(dropna=False)
    .sort_index()
    .rename("count")
)

print("final_df 클러스터별 수:")
print(cluster_counts)

final_df 클러스터별 수:
Cluster
1    3568
2    3480
3     334
Name: count, dtype: Int64


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================
# Repeated stratified bootstrap + mixed sampling
# avg_daily_steps: lognormal, others: normal
# =========================
N_BOOT = 5000
N_SAMPLES_PER_SEQN = 30
RANDOM_SEED = 42
OUT_PATH = "bootstrap_samples_5000x30.csv"
PROGRESS_EVERY = 500

sleep_time_vars = ["SLD012", "SLD013", "DAILY_SLEEP"]
sleep_pattern_vars = ["SLQ300", "SLQ310", "SLQ320", "SLQ330"]

# 층화 비율(고정): Cluster 1:2:3 = 65:26:9
cluster_ratio_by_cluster = {1: 65, 2: 26, 3: 9}
ratio_sum = sum(cluster_ratio_by_cluster.values())
if ratio_sum <= 0:
    raise ValueError("cluster_ratio_by_cluster 합계가 0보다 커야 합니다.")

# 총 층화 표본 수는 기존 설정(가능하면 sample_size_by_cluster 합계) 유지
if "sample_size_by_cluster" in globals():
    total_strata_n = int(sum(int(v) for v in sample_size_by_cluster.values() if int(v) > 0))
else:
    total_strata_n = 100

if total_strata_n <= 0:
    raise ValueError("총 층화 표본 수(total_strata_n)가 0보다 커야 합니다.")

# 비율 -> 정수 표본 수 변환(내림 + 큰 소수점 순으로 잔여 분배)
raw_target = {
    label: (total_strata_n * ratio) / ratio_sum
    for label, ratio in cluster_ratio_by_cluster.items()
}
target_n_by_cluster = {label: int(np.floor(v)) for label, v in raw_target.items()}
remainder = total_strata_n - sum(target_n_by_cluster.values())
if remainder > 0:
    frac_order = sorted(
        raw_target.keys(),
        key=lambda x: raw_target[x] - np.floor(raw_target[x]),
        reverse=True,
    )
    for label in frac_order[:remainder]:
        target_n_by_cluster[label] += 1

if "final_df" not in globals() or final_df.empty:
    raise ValueError("final_df is missing. Run the final_df merge cell first.")
if not {"SEQN", "CYCLE", "Cluster"}.issubset(final_df.columns):
    raise ValueError("final_df must include SEQN, CYCLE, Cluster.")

merged_base = final_df.copy()
merged_base["SEQN"] = pd.to_numeric(merged_base["SEQN"], errors="coerce")
merged_base["Cluster"] = pd.to_numeric(merged_base["Cluster"], errors="coerce").astype("Int64")
merged_base["CYCLE"] = merged_base["CYCLE"].astype(str)
merged_base = merged_base.dropna(subset=["SEQN", "Cluster", "CYCLE"]).copy()

# Build step score table (avg_daily_steps)
step_csv_path = Path("data_steps") / "csv" / "nhanes_1440_scsslsteps.csv"
if not step_csv_path.exists():
    raise FileNotFoundError(f"Step file not found: {step_csv_path}")

step_raw = pd.read_csv(step_csv_path)
if "SEQN" not in step_raw.columns:
    raise ValueError("Step file must include SEQN.")

step_cols = [c for c in step_raw.columns if c.startswith("step_min_")]
if not step_cols:
    step_cols = [c for c in step_raw.columns if c.startswith("min_")]
if not step_cols:
    step_cols = [c for c in step_raw.columns if c.startswith("step_")]
if not step_cols:
    candidate_cols = [c for c in step_raw.columns if c not in {"SEQN", "PAXDAYM", "PAXDAYWM"}]
    numeric_candidates = []
    for c in candidate_cols:
        converted = pd.to_numeric(step_raw[c], errors="coerce")
        if converted.notna().mean() >= 0.7:
            numeric_candidates.append(c)
    step_cols = numeric_candidates

if not step_cols:
    raise ValueError("No numeric step columns were found for avg_daily_steps.")

for c in step_cols:
    step_raw[c] = pd.to_numeric(step_raw[c], errors="coerce")

step_raw["daily_total_steps"] = step_raw[step_cols].sum(axis=1, skipna=True)
step_score_df = (
    step_raw.groupby("SEQN", as_index=False)["daily_total_steps"]
    .mean()
    .rename(columns={"daily_total_steps": "avg_daily_steps"})
)
step_score_df["SEQN"] = pd.to_numeric(step_score_df["SEQN"], errors="coerce")

merged_base = merged_base.merge(step_score_df, on="SEQN", how="left")

sleep_time_vars = [c for c in sleep_time_vars if c in merged_base.columns]
sleep_pattern_vars = [c for c in sleep_pattern_vars if c in merged_base.columns]
step_score_vars = ["avg_daily_steps"] if "avg_daily_steps" in merged_base.columns else []

sample_value_cols = sleep_time_vars + sleep_pattern_vars + step_score_vars
if not sample_value_cols:
    raise ValueError("No source columns available for bootstrap sampling.")

for c in sample_value_cols:
    merged_base[c] = pd.to_numeric(merged_base[c], errors="coerce")

if "RIDAGEYR" in merged_base.columns:
    merged_base["RIDAGEYR"] = pd.to_numeric(merged_base["RIDAGEYR"], errors="coerce")
else:
    merged_base["RIDAGEYR"] = np.nan

cluster_pools = {
    int(label): merged_base[merged_base["Cluster"] == int(label)].copy()
    for label in cluster_ratio_by_cluster.keys()
}

rng = np.random.default_rng(RANDOM_SEED)
out_path = Path(OUT_PATH)
if out_path.exists():
    out_path.unlink()


def _safe_sigma(values: pd.Series) -> tuple[float, float]:
    clean = pd.to_numeric(values, errors="coerce").dropna()
    if clean.empty:
        return np.nan, np.nan
    mu = float(clean.mean())
    sigma = float(clean.std(ddof=0))
    if not np.isfinite(sigma) or sigma == 0.0:
        sigma = abs(mu) * 0.05 if mu != 0 else 1.0
    return mu, sigma


def _safe_lognorm_params(values: pd.Series) -> tuple[float, float]:
    clean = pd.to_numeric(values, errors="coerce").dropna()
    clean = clean[clean > 0]
    if clean.empty:
        return np.nan, np.nan
    log_vals = np.log(clean)
    mu_log = float(log_vals.mean())
    sigma_log = float(log_vals.std(ddof=0))
    if not np.isfinite(sigma_log) or sigma_log == 0.0:
        sigma_log = 0.1
    return mu_log, sigma_log


written = False
total_rows = 0
preview_df = None
last_col_stats = {}

for boot_id in range(1, N_BOOT + 1):
    sampled_parts = []

    # 층화부트스트랩: 클러스터별 목표 수를 비율 기준으로 고정하고, 복원추출(replace=True)
    for cluster_label, target_size in target_n_by_cluster.items():
        if target_size <= 0:
            continue

        pool = cluster_pools.get(int(cluster_label), pd.DataFrame())
        if pool.empty:
            continue

        chosen_idx = rng.choice(pool.index.to_numpy(), size=int(target_size), replace=True)
        sampled_parts.append(pool.loc[chosen_idx])

    if not sampled_parts:
        continue

    # sampled_df = pd.concat(sampled_parts, ignore_index=True)
    # sampled_df = sampled_df[["SEQN", "CYCLE", "Cluster", "RIDAGEYR", *sample_value_cols]].copy()
    sampled_df = sampled_df.drop_duplicates(subset=["SEQN", "CYCLE"]).reset_index(drop=True)

    # 샘플 생성 기준 통계(부트스트랩 반복별)
    col_stats = {}
    for col in sample_value_cols:
        if col == "avg_daily_steps":
            mu_log, sigma_log = _safe_lognorm_params(sampled_df[col])
            if np.isfinite(mu_log) and np.isfinite(sigma_log):
                col_stats[col] = {"dist": "lognormal", "mu": mu_log, "sigma": sigma_log}
        else:
            mu, sigma = _safe_sigma(sampled_df[col])
            if np.isfinite(mu) and np.isfinite(sigma):
                col_stats[col] = {"dist": "normal", "mu": mu, "sigma": sigma}

    if not col_stats:
        continue

    n_seqn = len(sampled_df)
    out_df = pd.DataFrame({
        "bootstrap_id": np.repeat(boot_id, n_seqn * N_SAMPLES_PER_SEQN),
        "SEQN": np.repeat(sampled_df["SEQN"].to_numpy(dtype=float), N_SAMPLES_PER_SEQN),
        "CYCLE": np.repeat(sampled_df["CYCLE"].to_numpy(dtype=object), N_SAMPLES_PER_SEQN),
        "Cluster": np.repeat(sampled_df["Cluster"].to_numpy(dtype=int), N_SAMPLES_PER_SEQN),
        "RIDAGEYR": np.repeat(sampled_df["RIDAGEYR"].to_numpy(dtype=float), N_SAMPLES_PER_SEQN),
        "sample_idx": np.tile(np.arange(1, N_SAMPLES_PER_SEQN + 1), n_seqn),
    })
    out_df.insert(1, "bootstrap_row_id", np.arange(1, len(out_df) + 1))

    # 원 데이터 컬럼 단위로 샘플 생성 (걸음수=로그정규, 나머지=정규)
    for col, cfg in col_stats.items():
        if cfg["dist"] == "lognormal":
            out_df[col] = rng.lognormal(mean=cfg["mu"], sigma=cfg["sigma"], size=len(out_df))
        else:
            out_df[col] = rng.normal(loc=cfg["mu"], scale=cfg["sigma"], size=len(out_df))

    out_df.to_csv(
        out_path,
        mode="a" if written else "w",
        header=not written,
        index=False,
        encoding="utf-8-sig",
    )

    written = True
    total_rows += len(out_df)
    last_col_stats = col_stats

    if preview_df is None:
        preview_df = out_df.head().copy()

    if boot_id % PROGRESS_EVERY == 0:
        print(f"{boot_id}/{N_BOOT} done | rows so far: {total_rows}")

if not written:
    raise ValueError("No output rows were generated. Check input pools and required columns.")

max_rows_per_boot = (
    sum(
        int(target_n_by_cluster[label])
        for label in target_n_by_cluster
        if int(target_n_by_cluster[label]) > 0 and not cluster_pools.get(int(label), pd.DataFrame()).empty
    )
    * N_SAMPLES_PER_SEQN
)

print("Saved:", out_path)
print("step column count:", len(step_cols))
print("cluster ratio(1:2:3):", cluster_ratio_by_cluster)
print("total strata n per bootstrap:", total_strata_n)
print("cluster target sizes:", target_n_by_cluster)
print("bootstrap iterations:", N_BOOT)
print("samples per SEQN:", N_SAMPLES_PER_SEQN)
print("max rows per iteration (given pool limits):", max_rows_per_boot)
print("sample value columns:", list(last_col_stats.keys()) if written else [])
if "avg_daily_steps" in last_col_stats:
    print("avg_daily_steps 분포: lognormal")
print("RIDAGEYR included in meta columns")
print("total rows:", total_rows)
print(preview_df)


C:\Users\user\AppData\Local\Temp\ipykernel_26320\4160680354.py:89: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  step_raw["daily_total_steps"] = step_raw[step_cols].sum(axis=1, skipna=True)


500/5000 done | rows so far: 1500000
1000/5000 done | rows so far: 3000000
1500/5000 done | rows so far: 4500000
2000/5000 done | rows so far: 6000000
2500/5000 done | rows so far: 7500000
3000/5000 done | rows so far: 9000000
3500/5000 done | rows so far: 10500000
4000/5000 done | rows so far: 12000000
4500/5000 done | rows so far: 13500000
5000/5000 done | rows so far: 15000000
Saved: bootstrap_samples_5000x30.csv
step column count: 1440
cluster ratio(1:2:3): {1: 65, 2: 26, 3: 9}
total strata n per bootstrap: 100
cluster target sizes: {1: 65, 2: 26, 3: 9}
bootstrap iterations: 5000
samples per SEQN: 30
max rows per iteration (given pool limits): 3000
sample value columns: ['SLD012', 'SLD013', 'DAILY_SLEEP', 'SLQ300', 'SLQ310', 'SLQ320', 'SLQ330', 'avg_daily_steps']
avg_daily_steps 분포: lognormal
RIDAGEYR included in meta columns
total rows: 15000000
   bootstrap_id  bootstrap_row_id     SEQN CYCLE  Cluster  RIDAGEYR  \
0             1                 1  79005.0     H        1      44.

### fullX30.csv

In [10]:
# 전체 데이터 분포 기반 샘플링 — 30일치 생성 후 CSV 저장
# - 층화 부트스트래핑 없음 (전체 데이터 1회 집계 파라미터 사용)
# - 파일명: data_fullX30.csv
# - 분포 규칙은 bootstrap 셀과 동일: avg_daily_steps=로그정규, 나머지=정규

from pathlib import Path
import numpy as np
import pandas as pd

N_SAMPLES_PER_SEQN = 30
SEED = 42
OUT_PATH = Path("data_fullX30.csv")

# 소스 데이터 선택(merged_base 우선, 없으면 final_df)
if "merged_base" in globals() and isinstance(merged_base, pd.DataFrame) and not merged_base.empty:
    src_df = merged_base.copy()
elif "final_df" in globals() and isinstance(final_df, pd.DataFrame) and not final_df.empty:
    src_df = final_df.copy()
else:
    raise ValueError("데이터 소스(merged_base 또는 final_df)가 없습니다. 먼저 해당 셀들을 실행하세요.")

# 식별자 컬럼 확보
id_cols = [c for c in ["SEQN", "CYCLE", "Cluster", "RIDAGEYR"] if c in src_df.columns]
if "SEQN" not in id_cols:
    raise ValueError("SEQN 컬럼이 필요합니다 (데이터 소스에 SEQN 컬럼이 없습니다).")

# avg_daily_steps가 없으면 bootstrap 셀과 동일하게 data_steps에서 계산/병합
if "avg_daily_steps" not in src_df.columns:
    step_csv_path = Path("data_steps") / "csv" / "nhanes_1440_scsslsteps.csv"
    if not step_csv_path.exists():
        raise FileNotFoundError(f"걸음수 파일을 찾을 수 없습니다: {step_csv_path}")

    step_raw = pd.read_csv(step_csv_path)
    if "SEQN" not in step_raw.columns:
        raise ValueError("걸음수 파일에 SEQN 컬럼이 없습니다.")

    step_cols = [c for c in step_raw.columns if c.startswith("step_min_")]
    if not step_cols:
        step_cols = [c for c in step_raw.columns if c.startswith("min_")]
    if not step_cols:
        step_cols = [c for c in step_raw.columns if c.startswith("step_")]
    if not step_cols:
        candidate_cols = [c for c in step_raw.columns if c not in {"SEQN", "PAXDAYM", "PAXDAYWM"}]
        numeric_candidates = []
        for c in candidate_cols:
            converted = pd.to_numeric(step_raw[c], errors="coerce")
            if converted.notna().mean() >= 0.7:
                numeric_candidates.append(c)
        step_cols = numeric_candidates

    if not step_cols:
        raise ValueError("걸음수 파일에서 avg_daily_steps 계산용 수치형 컬럼을 찾지 못했습니다.")

    for c in step_cols:
        step_raw[c] = pd.to_numeric(step_raw[c], errors="coerce")

    step_raw["daily_total_steps"] = step_raw[step_cols].sum(axis=1, skipna=True)
    step_score_df = (
        step_raw.groupby("SEQN", as_index=False)["daily_total_steps"]
        .mean()
        .rename(columns={"daily_total_steps": "avg_daily_steps"})
    )
    step_score_df["SEQN"] = pd.to_numeric(step_score_df["SEQN"], errors="coerce")

    src_df["SEQN"] = pd.to_numeric(src_df["SEQN"], errors="coerce")
    src_df = src_df.merge(step_score_df, on="SEQN", how="left")

# 샘플링 대상 컬럼(가능한 컬럼만 선택)
sleep_time_vars = ["SLD012", "SLD013", "DAILY_SLEEP"]
sleep_pattern_vars = ["SLQ300", "SLQ310", "SLQ320", "SLQ330"]
step_score_vars = ["avg_daily_steps"]

sample_value_cols = [c for c in (sleep_time_vars + sleep_pattern_vars + step_score_vars) if c in src_df.columns]
if not sample_value_cols:
    raise ValueError("샘플링할 수치형 컬럼이 없습니다. (예: DAILY_SLEEP, SLQ300, avg_daily_steps 등)")

# 분포 파라미터 계산 (avg_daily_steps=로그정규, 나머지=정규)
def _safe_sigma(series: pd.Series) -> tuple[float, float]:
    clean = pd.to_numeric(series, errors="coerce").dropna()
    if clean.empty:
        return (np.nan, np.nan)
    mu = float(clean.mean())
    sigma = float(clean.std(ddof=0))
    if not np.isfinite(sigma) or sigma == 0.0:
        sigma = abs(mu) * 0.05 if mu != 0 else 1.0
    return mu, sigma

def _safe_lognorm_params(series: pd.Series) -> tuple[float, float]:
    clean = pd.to_numeric(series, errors="coerce").dropna()
    clean = clean[clean > 0]
    if clean.empty:
        return (np.nan, np.nan)
    log_vals = np.log(clean)
    mu_log = float(log_vals.mean())
    sigma_log = float(log_vals.std(ddof=0))
    if not np.isfinite(sigma_log) or sigma_log == 0.0:
        sigma_log = 0.1
    return mu_log, sigma_log

col_stats = {}
for col in sample_value_cols:
    if col == "avg_daily_steps":
        mu_log, sigma_log = _safe_lognorm_params(src_df[col])
        if np.isfinite(mu_log) and np.isfinite(sigma_log):
            col_stats[col] = {"dist": "lognormal", "mu": mu_log, "sigma": sigma_log}
    else:
        mu, sigma = _safe_sigma(src_df[col])
        if np.isfinite(mu) and np.isfinite(sigma):
            col_stats[col] = {"dist": "normal", "mu": mu, "sigma": sigma}

if not col_stats:
    raise ValueError("유효한 분포 파라미터를 계산할 수 있는 컬럼이 없습니다.")

# 중복(SEQN,CYCLE) 제거한 ID 테이블 준비
dup_subset = [c for c in ["SEQN", "CYCLE"] if c in src_df.columns]
id_table = src_df[id_cols].drop_duplicates(subset=dup_subset).copy()
# SEQN 숫자형 보장
id_table["SEQN"] = pd.to_numeric(id_table["SEQN"], errors="coerce")
id_table = id_table.dropna(subset=["SEQN"]).reset_index(drop=True)

# 확장(각 SEQN 당 N_SAMPLES_PER_SEQN)
expanded = id_table.loc[id_table.index.repeat(N_SAMPLES_PER_SEQN)].reset_index(drop=True)
expanded.insert(0, "sample_idx", np.tile(np.arange(1, N_SAMPLES_PER_SEQN + 1), len(id_table)))

rng = np.random.default_rng(SEED)
for col, cfg in col_stats.items():
    if cfg["dist"] == "lognormal":
        expanded[col] = rng.lognormal(mean=cfg["mu"], sigma=cfg["sigma"], size=len(expanded))
    else:
        expanded[col] = rng.normal(loc=cfg["mu"], scale=cfg["sigma"], size=len(expanded))

# 파일 저장
expanded.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

print(f"✅ 저장 완료: {OUT_PATH} | 행수: {len(expanded)} | 샘플/SEQN: {N_SAMPLES_PER_SEQN}")
print("샘플링 대상 컬럼:", list(col_stats.keys()))
if "avg_daily_steps" in col_stats:
    print("avg_daily_steps 분포: lognormal")
expanded.head()

C:\Users\user\AppData\Local\Temp\ipykernel_26248\99504337.py:57: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  step_raw["daily_total_steps"] = step_raw[step_cols].sum(axis=1, skipna=True)


✅ 저장 완료: data_fullX30.csv | 행수: 221460 | 샘플/SEQN: 30
샘플링 대상 컬럼: ['SLD012', 'SLD013', 'DAILY_SLEEP', 'SLQ300', 'SLQ310', 'SLQ320', 'SLQ330', 'avg_daily_steps']
avg_daily_steps 분포: lognormal


,sample_idx,SEQN,CYCLE,Cluster,RIDAGEYR,SLD012,SLD013,DAILY_SLEEP,SLQ300,SLQ310,SLQ320,SLQ330,avg_daily_steps
0,1,62161.0,G,1,22.0,7.679064,10.458876,6.588900,23.681405,9.533737,11.875966,9.544911,5801.193961
1,2,62161.0,G,1,22.0,5.624390,9.873856,6.847504,-7.500669,3.018559,2.859269,7.877547,9500.616649
2,3,62161.0,G,1,22.0,8.360136,6.818109,6.289475,-1.567515,5.380321,7.062781,2.961101,36969.475228
3,4,62161.0,G,1,22.0,8.650625,7.571874,7.795440,0.834471,7.550205,-2.692433,4.783470,14292.746217
4,5,62161.0,G,1,22.0,4.232324,9.554261,4.716888,2.442969,6.907663,9.313623,7.449218,6298.471813
